# TFT Scouting — Notebook 2: Match Fetcher

**Objetivo:** Para cada jugador del CSV anterior, descargar sus últimas N partidas
y guardar el JSON raw en Newline-Delimited JSON (NDJSON) para carga directa a BigQuery.

**Nota de routing:** match-v1 usa REGIONAL routing → `europe.api.riotgames.com`

**Estimación de llamadas para N=200 jugadores × 20 partidas:**
- 200 llamadas a matchlist
- ~4000 llamadas a match detail (deduplicadas serán menos)
- A 1 req/s conservador: ~70 minutos → perfecto para correr overnight

In [1]:
import os, time, json, requests
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path

API_KEY = 'RGAPI-c05d22b7-d146-4011-b5af-6307a0397f1a'
REGION  = 'europe.api.riotgames.com'
HEADERS = {'X-Riot-Token': API_KEY}

MATCHES_PER_PLAYER = 20  # Ajusta según necesidad
OUTPUT_FILE = 'matches_raw.ndjson'
SLEEP_BETWEEN_REQUESTS = 0.07  # ~14 req/s, bajo el límite de 20

In [2]:
def get(url, params=None, retries=3):
    for attempt in range(retries):
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code == 200:
            return r.json()
        elif r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 10))
            print(f'  Rate limit — esperando {wait}s')
            time.sleep(wait)
        elif r.status_code == 404:
            return None
        else:
            print(f'  Error {r.status_code} en {url}')
            time.sleep(2)
    return None

df_players = pd.read_csv('top_players.csv')
print(f'Jugadores cargados: {len(df_players)}')

Jugadores cargados: 200


In [3]:
# ── 1. Obtener match IDs por jugador ─────────────────────────────────────────

all_match_ids = set()
player_match_map = {}  # puuid → [match_ids]

for _, row in tqdm(df_players.iterrows(), total=len(df_players), desc='Match IDs'):
    puuid = row['puuid']
    url = f'https://{REGION}/tft/match/v1/matches/by-puuid/{puuid}/ids'
    match_ids = get(url, params={'count': MATCHES_PER_PLAYER, 'queue': 1100})  # 1100 = ranked
    if match_ids:
        player_match_map[puuid] = match_ids
        all_match_ids.update(match_ids)
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print(f'Match IDs únicos: {len(all_match_ids)}')

Match IDs:   0%|          | 0/200 [00:00<?, ?it/s]

  Rate limit — esperando 82s
Match IDs únicos: 2745


In [4]:
# ── 2. Descargar match details ────────────────────────────────────────────────
# Guardamos en NDJSON (una línea JSON por partida) para carga directa a BigQuery

already_fetched = set()

# Si el archivo ya existe (restart parcial), cargamos los IDs ya procesados
if Path(OUTPUT_FILE).exists():
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            match = json.loads(line)
            already_fetched.add(match['metadata']['match_id'])
    print(f'Ya descargadas: {len(already_fetched)} partidas')

pending = all_match_ids - already_fetched
print(f'Pendientes: {len(pending)}')

Pendientes: 2745


In [5]:
errors = []

with open(OUTPUT_FILE, 'a') as f_out:
    for match_id in tqdm(pending, desc='Descargando partidas'):
        url = f'https://{REGION}/tft/match/v1/matches/{match_id}'
        data = get(url)
        if data:
            f_out.write(json.dumps(data) + '\n')
        else:
            errors.append(match_id)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

print(f'Descarga completa. Errores: {len(errors)}')
if errors:
    print('Match IDs con error:', errors[:10])

Descargando partidas:   0%|          | 0/2745 [00:00<?, ?it/s]

  Rate limit — esperando 80s
  Rate limit — esperando 68s
  Rate limit — esperando 68s
  Rate limit — esperando 68s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 69s
  Rate limit — esperando 68s
  Rate limit — esperando 68s
  Rate limit — esperando 69s
  Rate limit — esperando 68s
  Rate limit — esperando 72s
  Rate limit — esperando 76s
  Rate limit — esperando 77s
  Rate limit — esperando 77s
  Rate limit — esperando 73s
  Rate limit — esperando 75s
  Rate limit — esperando 74s
  Rate limit — esperando 75s
  Rate limit — esperando 75s
  Rate limit — esperando 75s
  Rate limit — esperando 75s
  Rate limit — esperando 75s
Descarga completa. Errores: 0


In [6]:
# ── 3. Inspección rápida del JSON ─────────────────────────────────────────────
# Ver la estructura real antes de modelar BigQuery

with open(OUTPUT_FILE, 'r') as f:
    sample = json.loads(f.readline())

print('=== METADATA ===')
print(json.dumps(sample['metadata'], indent=2))

print('\n=== INFO (top level) ===')
print(list(sample['info'].keys()))

print('\n=== PARTICIPANT 0 (top level keys) ===')
p = sample['info']['participants'][0]
print(list(p.keys()))

print('\n=== PARTICIPANT 0 — UNITS ===')
print(json.dumps(p['units'][:2], indent=2))

print('\n=== PARTICIPANT 0 — AUGMENTS ===')
print(p.get('augments', []))

print('\n=== PARTICIPANT 0 — TRAITS ===')
print(json.dumps(p['traits'][:3], indent=2))

=== METADATA ===
{
  "data_version": "6",
  "match_id": "EUW1_7761291413",
  "participants": [
    "UyM8O0EJnOuxoE9IR3JUkQWjK076_cW_Ku9OLOCHHK-kek3fCu0ROSiAu_Rq5eZwbgR7_ANr9hA__g",
    "cIS7pCMhM9ugJZcxVqpKmFxJH3qNNX4JKefGin6ZKa91_NBOyfopNVcmuVIUa0RgULkq17FYLE4KJA",
    "Cup1XEioQGQbYxzY9sAPFEcJUbXLJj_CTD0QxM9EYpHejjcjpzrVSzFgCtK0_7Q0POgsvtjAvCwUrw",
    "NZbnJ3Jib9qdx1NV-XsNG1BujMbpeWH2v4LnDZCRuRBazZoH7wNTUFPdLgz9txRPu8W2VnTgEazi-A",
    "-KZi-fjYvVUbig-wOXzkzZ_msw5GU8PbkgJ2qm_idP8GHWKYYzM97aOLuAk6mzRn5PgZU7Qi-jbt-A",
    "0K-z364xHFQlhFcGJqLrJLuL1FfnQcIJU_fFkovA_ns_Z7ziyVoh_c7za4vwrbNsk-Mo4e-cNQDO0Q",
    "ztxLTw0KALvo1OoiMsi5V7uzMYmr0fVh10BFvBjW-lmRyu8gu_mxQcuL9-dFV2wonWSz5vIrnAz3sQ",
    "tSghZRAB5lDNWHdARBrv1LGgPFAy4QzpH9pMCaAt-R_qgb1RDbtkjEbFWkoHQNNqdRJpSmC9QBvFdw"
  ]
}

=== INFO (top level) ===
['endOfGameResult', 'gameCreation', 'gameId', 'game_datetime', 'game_length', 'game_version', 'mapId', 'participants', 'queueId', 'queue_id', 'tft_game_type', 'tft_set_core_name', 'tft_s

In [7]:
# ── 4. Resumen estadístico rápido ─────────────────────────────────────────────

matches = []
with open(OUTPUT_FILE, 'r') as f:
    for line in f:
        matches.append(json.loads(line))

print(f'Total partidas descargadas: {len(matches)}')
print(f'Participantes por partida: {len(matches[0]["info"]["participants"])}')

# Placements de muestra
placements = [p['placement'] for m in matches[:100] for p in m['info']['participants']]
print(f'Placements en primeras 100 partidas: {sorted(set(placements))}')

Total partidas descargadas: 2745
Participantes por partida: 8
Placements en primeras 100 partidas: [1, 2, 3, 4, 5, 6, 7, 8]
